# ProspectRL — Stage 1: Learn to Mine

**Goal:** Teach the agent to navigate toward ore and dig it. No fuel management, no caves, no mixed preferences — just pure ore-seeking behavior.

| Parameter | Value |
|-----------|-------|
| World size | 20 x 40 x 20 |
| Ore density | 10x (very dense) |
| Fuel | Infinite |
| Caves | OFF |
| Preferences | One-hot (single ore per episode) |
| Max steps | 500 |
| Advancement | mean_reward > 50.0 over 100 episodes |

**What success looks like:** The agent consistently moves toward the nearest ore of the requested type and digs it, rather than wandering randomly. `mining/ores_per_episode` should reach 5-10+ and `rollout/ep_rew_mean` should exceed 50.

In [ ]:
import subprocess
import sys

# GitHub repo — update this with your actual repo URL
GITHUB_REPO = "https://github.com/YOUR_USERNAME/ProspectRL.git"

# Clone and install (use [colab] extra — [all] includes viz deps that need system GL libs)
subprocess.run(
    ["git", "clone", "--depth", "1", GITHUB_REPO, "/content/prospect_rl"],
    check=True,
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "/content/prospect_rl[colab]"]
)
print("Installed from GitHub")

In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')

# ── Stage 1 Configuration ──────────────────────────────────────────
STAGE = 0           # 0 = stage1_dense_easy (20x40x20, 10x ore, infinite fuel)
N_ENVS = 4          # Parallel environments
TOTAL_TIMESTEPS = 500_000   # Stage 1 converges fast due to dense ores
SEED = 42
CHECKPOINT_DIR = "/content/drive/MyDrive/prospect_rl_checkpoints/stage1"
RESUME = True       # Resumes from latest checkpoint if one exists

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Print Stage 1 summary
from prospect_rl.config import CURRICULUM_STAGES
stage_cfg = CURRICULUM_STAGES[STAGE]
sx, sy, sz = stage_cfg.world_size

print("=" * 50)
print(f"  STAGE 1: {stage_cfg.name}")
print(f"  World:       {sx} x {sy} x {sz}")
print(f"  Ore density: {stage_cfg.ore_density_multiplier}x")
print(f"  Fuel:        {'infinite' if stage_cfg.infinite_fuel else stage_cfg.max_fuel}")
print(f"  Caves:       {'ON' if stage_cfg.caves_enabled else 'OFF'}")
print(f"  Preferences: {stage_cfg.preference_mode}")
print(f"  Max steps:   {stage_cfg.max_episode_steps}")
print(f"  Target:      {stage_cfg.advancement_metric} > {stage_cfg.advancement_threshold}")
print("=" * 50)
print(f"\n  Timesteps:   {TOTAL_TIMESTEPS:,}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
import json
from pathlib import Path

from prospect_rl.config import Config
from prospect_rl.models.callbacks import (
    CurriculumCallback,
    DriveCheckpointCallback,
    MetricsCallback,
)
from prospect_rl.models.ppo_config import create_ppo_model, make_training_env
from sb3_contrib import MaskablePPO
from stable_baselines3.common.callbacks import CallbackList
from stable_baselines3.common.vec_env import VecNormalize

# Create vectorized training environment
env = make_training_env(n_envs=N_ENVS, stage_index=STAGE, seed=SEED)

# Show observation and action spaces
print("Observation space:")
for key, space in env.observation_space.spaces.items():
    print(f"  {key:>8}: shape={space.shape}  dtype={space.dtype}")
print(f"\nAction space: Discrete({env.action_space.n})")
print(f"  actions: forward, back, up, down, turn_left, turn_right, dig, dig_up, dig_down")

In [ ]:
config = Config()
reset_num_timesteps = True

if RESUME:
    latest_path = Path(CHECKPOINT_DIR) / "latest.json"
    if latest_path.exists():
        manifest = json.loads(latest_path.read_text())
        print(f"Resuming from step {manifest['step']}")
        env = VecNormalize.load(manifest["vecnormalize_path"], env.venv)
        env.training = True
        env.norm_reward = True
        model = MaskablePPO.load(manifest["model_path"], env=env)
        reset_num_timesteps = False
    else:
        print("No checkpoint found, starting fresh")
        model = create_ppo_model(env, config, tensorboard_log="./tb_logs", seed=SEED)
else:
    model = create_ppo_model(env, config, tensorboard_log="./tb_logs", seed=SEED)

## Training

**What to watch in TensorBoard:**
- `rollout/ep_rew_mean` — main signal, target > 50
- `mining/ores_per_episode` — should climb from ~0 to 5-10+
- `mining/episode_steps` — if episodes max out at 500, the agent is exploring (good early); if they shorten, it may be dying or getting stuck

Stage 1 with 500k timesteps typically takes 15-30 minutes on a Colab T4 GPU.

In [ ]:
callbacks = CallbackList([
    DriveCheckpointCallback(
        checkpoint_dir=CHECKPOINT_DIR,
        save_freq=config.training.checkpoint_freq,
        max_kept=config.training.max_checkpoints_kept,
        verbose=1,
    ),
    MetricsCallback(),
    CurriculumCallback(stage_index=STAGE),
])

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callbacks,
    reset_num_timesteps=reset_num_timesteps,
)

# Save final
final_dir = Path(CHECKPOINT_DIR) / "final"
final_dir.mkdir(parents=True, exist_ok=True)
model.save(str(final_dir / "model.zip"))
env.save(str(final_dir / "vecnormalize.pkl"))
print("Training complete!")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./tb_logs

## Evaluation

Stage 1 uses **one-hot preferences** — each evaluation episode targets a single ore type. If the agent scores well across all ore types, it has learned general ore-seeking behavior and is ready for Stage 2.

In [ ]:
from prospect_rl.evaluate import evaluate

results = evaluate(
    model_path=str(final_dir / "model.zip"),
    stage_index=STAGE,
    n_episodes=10,
)

print(f"\n{'Preference':<25} {'Reward':>10} {'Ores':>8} {'Steps':>8}")
print("-" * 55)
for name, metrics in results.items():
    marker = " <<" if metrics["mean_reward"] > 50 else ""
    print(
        f"{name:<25} "
        f"{metrics['mean_reward']:>10.2f} "
        f"{metrics['mean_ores']:>8.1f} "
        f"{metrics['mean_steps']:>8.1f}{marker}"
    )

passing = sum(1 for m in results.values() if m["mean_reward"] > 50)
print(f"\n{passing}/{len(results)} preferences above advancement threshold (50.0)")
if passing == len(results):
    print("Ready for Stage 2!")

## Export

This exports a Stage 1 checkpoint. It is **not deployment-ready** — the agent hasn't learned fuel management or cave navigation yet. Use it as a warm-start for Stage 2.

In [ ]:
from prospect_rl.deployment.model_export import export_for_deployment

export_dir = Path(CHECKPOINT_DIR) / "export"
export_for_deployment(
    model_path=str(final_dir / "model.zip"),
    vecnormalize_path=str(final_dir / "vecnormalize.pkl"),
    output_dir=str(export_dir),
)
print(f"Model exported to {export_dir}")

## Next: Stage 2

Stage 2 introduces **fuel management** in a larger world:

| Parameter | Stage 1 | Stage 2 |
|-----------|---------|---------|
| World | 20x40x20 | 32x64x32 |
| Ore density | 10x | 3x |
| Fuel | infinite | 500 |
| Preferences | one-hot | one-hot |
| Max steps | 500 | 800 |

To continue training, duplicate this notebook and set `STAGE = 1`. The agent will load its Stage 1 weights and learn to mine efficiently under fuel pressure.